In [1]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.memory_allocated() / 1024**2, "MB")
print(torch.cuda.memory_reserved() / 1024**2, "MB")

True
0.0 MB
0.0 MB


### Importing Libraries

In [2]:
import os
import re
import random
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split

### Device and Random Seed


In [3]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(device)

cuda


### Paths

In [4]:
ROOT = Path(r"E:\friends")

TEXT_ROOT = ROOT / "transcripts"

VIDEO_ROOT = ROOT / "video_windows"

FMRI_ROOTS = {

    "sub1": ROOT / "fmri" / "preprocessed_sub1_npy",

    "sub2": ROOT / "fmri" / "preprocessed_sub2_npy",

    "sub3": ROOT / "fmri" / "preprocessed_sub3_npy",

    "sub5": ROOT / "fmri" / "preprocessed_sub5_npy"

}

TRAIN_SEASONS = [

    "s1",
    "s2",
    "s3",
    "s4",
    "s5"

]

TEST_SEASON = "s6"

### Synchronize the Text + Video with fMRI data

In [5]:
def episode_name(path):

    match = re.search(

        r"s\d{2}e\d{2}[a-z]?",

        path.stem

    )

    if match is None:

        raise ValueError(path.name)

    return match.group(0)


def get_fmri_file(folder, episode):

    files = list(

        folder.glob(f"*{episode}.npy")

    )

    if len(files) == 0:

        return None

    return files[0]

# windows of 8 words, with stride 1
def build_text_windows(text):

    windows = []

    for i in range(len(text)-7):

        windows.append(

            text[i:i+8]

        )

    return np.stack(windows)

### One training sample for every valid TR

In [6]:
train_index = []

for season in TRAIN_SEASONS:

    text_folder = TEXT_ROOT / season

    video_folder = VIDEO_ROOT / season

    text_files = sorted(

        text_folder.glob("*_llama.npy")

    )

    for text_file in tqdm(

        text_files,

        desc=f"{season}"

    ):

        episode = episode_name(text_file)

        video_file = video_folder / (

            text_file.name.replace(

                "_llama.npy",

                "_video_windows.npy"

            )

        )

        if not video_file.exists():

            continue

        for subject, fmri_root in FMRI_ROOTS.items():

            fmri_file = get_fmri_file(

                fmri_root / "s1-s5",

                episode

            )

            if fmri_file is None:

                continue

            text = np.load(text_file,mmap_mode="r")

            video = np.load(video_file,mmap_mode="r")

            fmri = np.load(fmri_file,mmap_mode="r")

            usable = min(

                len(text)-7,

                len(video),

                len(fmri)-5

            )

            for tr in range(usable):

                train_index.append({

                    "text":text_file,

                    "video":video_file,

                    "fmri":fmri_file,

                    "tr":tr

                })

print("Training Samples:",len(train_index))

s1:   0%|          | 0/47 [00:00<?, ?it/s]

s2:   0%|          | 0/48 [00:00<?, ?it/s]

s3:   0%|          | 0/50 [00:00<?, ?it/s]

s4:   0%|          | 0/48 [00:00<?, ?it/s]

s5:   0%|          | 0/48 [00:00<?, ?it/s]

Training Samples: 446580


### One testing sample per TR


In [7]:
test_index = []

text_folder = TEXT_ROOT / TEST_SEASON

video_folder = VIDEO_ROOT / TEST_SEASON

text_files = sorted(

    text_folder.glob("*_llama.npy")

)

for text_file in tqdm(

    text_files,

    desc="Testing"

):

    episode = episode_name(text_file)

    video_file = video_folder / (

        text_file.name.replace(

            "_llama.npy",

            "_video_windows.npy"

        )

    )

    if not video_file.exists():

        continue

    for subject, fmri_root in FMRI_ROOTS.items():

        fmri_file = get_fmri_file(

            fmri_root/"s6",

            episode

        )

        if fmri_file is None:

            continue

        text = np.load(text_file,mmap_mode="r")

        video = np.load(video_file,mmap_mode="r")

        fmri = np.load(fmri_file,mmap_mode="r")

        usable = min(

            len(text)-7,

            len(video),

            len(fmri)-5

        )

        for tr in range(usable):

            test_index.append({

                "text":text_file,

                "video":video_file,

                "fmri":fmri_file,

                "tr":tr

            })

print("Testing Samples:",len(test_index))

Testing:   0%|          | 0/50 [00:00<?, ?it/s]

Testing Samples: 91665


In [8]:
train_index, val_index = train_test_split(

    train_index,

    test_size=0.1,

    random_state=42,

    shuffle=True

)

print(len(train_index))

print(len(val_index))

print(len(test_index))

401922
44658
91665


### Build dataset

In [9]:
class BrainDataset(Dataset):

    def __init__(self, index):
        self.index = index

    def __len__(self):
        return len(self.index)

    def __getitem__(self, idx):

        item = self.index[idx]

        # Memory-map files (nothing loaded fully into RAM)
        text = np.load(
            item["text"],
            mmap_mode="r"
        )

        video = np.load(
            item["video"],
            mmap_mode="r"
        )

        fmri = np.load(
            item["fmri"],
            mmap_mode="r"
        )

        tr = item["tr"]

        # Build ONLY ONE text window
        text_window = np.asarray(
            text[tr:tr+8],
            dtype=np.float32
        )

        # One video window
        video_window = np.asarray(
            video[tr],
            dtype=np.float32
        )

        # Fuse modalities
        fused = np.concatenate(
            [
                text_window,
                video_window
            ],
            axis=-1
        )

        target = np.asarray(
            fmri[tr+5],
            dtype=np.float32
        )

        return (
            torch.from_numpy(fused),
            torch.from_numpy(target)
        )
# Fuse the text and video modalities. Converts both the fused input and target brain activity into PyTorch tensors

In [10]:
train_dataset = BrainDataset(train_index)
val_dataset = BrainDataset(val_index)
test_dataset = BrainDataset(test_index)

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

In [11]:
x,y=next(iter(train_loader))

print(x.shape)

print(y.shape)

C:\Users\milin\AppData\Local\Temp\ipykernel_20812\2407310007.py:59: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  torch.from_numpy(target)


torch.Size([8, 8, 2816])
torch.Size([8, 1000])


### Modality Dropout

In [12]:
class ModalityDropout(nn.Module):

    def __init__(self,p=0.15):

        super().__init__()

        self.p=p

    def forward(self,x):

        if not self.training:

            return x

        B=x.size(0)

        device=x.device

        text_keep=(torch.rand(B,device=device)>self.p).float()

        video_keep=(torch.rand(B,device=device)>self.p).float()

        x=x.clone()

        x[:,:,:2048]*=text_keep[:,None,None]

        x[:,:,2048:]*=video_keep[:,None,None]

        return x

### Positional Embedding

In [13]:
class PositionalEmbedding(nn.Module):

    def __init__(self,d_model,max_len=8):

        super().__init__()

        self.pos=nn.Parameter(

            torch.randn(

                1,

                max_len,

                d_model

            )

        )

    def forward(self,x):

        return x+self.pos

In [14]:
class CLSToken(nn.Module):

    def __init__(self,d_model):

        super().__init__()

        self.cls=nn.Parameter(

            torch.randn(

                1,

                1,

                d_model

            )

        )

    def forward(self,x):

        B=x.size(0)

        cls=self.cls.expand(

            B,

            -1,

            -1

        )

        return torch.cat(

            [

                cls,

                x

            ],

            dim=1

        )

In [15]:
class TribeLite(nn.Module):

    def __init__(self):

        super().__init__()

        self.modality_dropout=ModalityDropout(0.15)

        self.input_proj=nn.Sequential(

            nn.LayerNorm(2816),

            nn.Linear(2816,384),

            nn.GELU(),

            nn.Dropout(0.1)

        )

        self.pos=PositionalEmbedding(

            384,

            max_len=8

        )

        self.cls=CLSToken(384)

        encoder_layer=nn.TransformerEncoderLayer(

            d_model=384,

            nhead=8,

            dim_feedforward=1024,

            dropout=0.1,

            activation="gelu",

            batch_first=True

        )

        self.encoder=nn.TransformerEncoder(

            encoder_layer,

            num_layers=4

        )

        self.head=nn.Sequential(

            nn.LayerNorm(384),

            nn.Linear(

                384,

                768

            ),

            nn.GELU(),

            nn.Dropout(0.2),

            nn.Linear(

                768,

                1000

            )

        )

    def forward(self,x):

        x=self.modality_dropout(x)

        x=self.input_proj(x)

        x=self.pos(x)

        x=self.cls(x)

        x=self.encoder(x)

        cls=x[:,0]

        return self.head(cls)

In [16]:
device=torch.device(

    "cuda"

    if torch.cuda.is_available()

    else "cpu"

)

model=TribeLite().to(device)

print(

    "Parameters:",

    sum(

        p.numel()

        for p in model.parameters()

    )/1e6,

    "Million"

)

Parameters: 7.679208 Million


In [17]:
criterion=nn.SmoothL1Loss()

optimizer=torch.optim.AdamW(

    model.parameters(),

    lr=2e-4,

    weight_decay=1e-2

)

scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(

    optimizer,

    T_max=20

)

from torch.optim.swa_utils import AveragedModel,SWALR

swa_model=AveragedModel(model)

swa_scheduler=SWALR(

    optimizer,

    swa_lr=1e-4

)

In [18]:
scaler = torch.amp.GradScaler("cuda")

In [19]:
import numpy as np

def pearson_corr(pred, target):

    pred = pred - pred.mean(axis=0)

    target = target - target.mean(axis=0)

    numerator = (pred * target).sum(axis=0)

    denominator = np.sqrt(

        (pred**2).sum(axis=0)

        *

        (target**2).sum(axis=0)

    )

    corr = numerator / (denominator + 1e-8)

    return np.mean(corr)

In [20]:
from tqdm.auto import tqdm

def train_one_epoch(model, loader):

    model.train()

    running_loss = 0

    pbar = tqdm(loader)

    for x, y in pbar:

        x = x.to(device, non_blocking=True)

        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(

            "cuda",

            dtype=torch.float16

        ):

            pred = model(x)

            loss = criterion(pred, y)

        scaler.scale(loss).backward()

        torch.nn.utils.clip_grad_norm_(

            model.parameters(),

            1.0

        )

        scaler.step(optimizer)

        scaler.update()

        running_loss += loss.item()

        pbar.set_description(

            f"Loss {loss.item():.5f}"

        )

    scheduler.step()

    return running_loss / len(loader)

In [21]:
def validate(model, loader):

    model.eval()

    losses = []

    preds = []

    targets = []

    with torch.no_grad():

        for x, y in tqdm(loader):

            x = x.to(device)

            y = y.to(device)

            with torch.amp.autocast(

                "cuda",

                dtype=torch.float16

            ):

                pred = model(x)

                loss = criterion(pred, y)

            losses.append(loss.item())

            preds.append(

                pred.float().cpu().numpy()

            )

            targets.append(

                y.cpu().numpy()

            )

    preds = np.concatenate(preds)

    targets = np.concatenate(targets)

    corr = pearson_corr(

        preds,

        targets

    )

    return np.mean(losses), corr

In [ ]:
from pathlib import Path
import time

NUM_EPOCHS = 20
PATIENCE = 5

SAVE_DIR = Path(r"E:\friends\models")
SAVE_DIR.mkdir(exist_ok=True)

best_corr = -1.0
patience_counter = 0

history = {
    "train_loss": [],
    "val_loss": [],
    "pearson": []
}

for epoch in range(NUM_EPOCHS):

    print(f"\n{'='*60}")
    print(f"Epoch {epoch+1}/{NUM_EPOCHS}")
    print(f"{'='*60}")

    train_loss = train_one_epoch(
        model,
        train_loader
    )

    val_loss, corr = validate(
        model,
        val_loader
    )

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["pearson"].append(corr)

    print(f"\nTrain Loss : {train_loss:.5f}")
    print(f"Val Loss   : {val_loss:.5f}")
    print(f"Pearson    : {corr:.5f}")
    print(f"LR         : {optimizer.param_groups[0]['lr']:.2e}")

    if corr > best_corr:

        best_corr = corr
        patience_counter = 0

        save_path = SAVE_DIR / f"best_tribe_lite_epoch{epoch+1}.pth"

        torch.save(
            model.state_dict(),
            save_path
        )

        print(f"\n✅ Best Model Saved")
        print(save_path)

    else:

        patience_counter += 1
        print(f"\nNo Improvement ({patience_counter}/{PATIENCE})")

    if epoch >= 15:

        swa_model.update_parameters(model)
        swa_scheduler.step()

    if patience_counter >= PATIENCE:

        print("\nEarly Stopping Triggered")
        break

print("\nTraining Finished")
print(f"Best Pearson : {best_corr:.5f}")


Epoch 1/20


  0%|          | 0/50241 [00:00<?, ?it/s]

In [ ]:
from pathlib import Path

SAVE_DIR = Path(r"E:\friends\models")

for file in SAVE_DIR.glob("*.pth"):
    print(file.name)

best_tribe_lite_epoch1.pth
best_tribe_lite_epoch10.pth
best_tribe_lite_epoch11.pth
best_tribe_lite_epoch12.pth
best_tribe_lite_epoch13.pth
best_tribe_lite_epoch14.pth
best_tribe_lite_epoch15.pth
best_tribe_lite_epoch16.pth
best_tribe_lite_epoch17.pth
best_tribe_lite_epoch18.pth
best_tribe_lite_epoch19.pth
best_tribe_lite_epoch2.pth
best_tribe_lite_epoch20.pth
best_tribe_lite_epoch3.pth
best_tribe_lite_epoch4.pth
best_tribe_lite_epoch5.pth
best_tribe_lite_epoch6.pth
best_tribe_lite_epoch7.pth
best_tribe_lite_epoch8.pth
best_tribe_lite_epoch9.pth


In [ ]:
model.load_state_dict(
    torch.load(
        r"E:\friends\models\best_tribe_lite_epoch20.pth",
        map_location=device
    )
)

model.eval()

print("Best model loaded successfully!")

Best model loaded successfully!


In [ ]:
test_loss, test_corr = validate(
    model,
    test_loader
)

print(f"Test Loss      : {test_loss:.5f}")
print(f"Test Pearson   : {test_corr:.5f}")

  0%|          | 0/22917 [00:00<?, ?it/s]

Test Loss      : 0.40059
Test Pearson   : 0.20048
